# Kannada SSL — Exploratory Data Analysis

This notebook explores the Kannada datasets used for self-supervised learning.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import librosa
import librosa.display
import os

from config import Config
from utils.kannada_utils import load_padakosha, load_classification_dataset
from utils.audio_utils import load_audio, list_kannada_audio_files, get_audio_stats

config = Config()
plt.style.use('dark_background')
print('Setup complete!')

## 1. Audio Dataset Analysis

In [ ]:
audio_files = list_kannada_audio_files(config.AUDIO_DATASET_DIR)
print(f'Total audio files: {len(audio_files)}')
print('First 5 files:')
for f in audio_files[:5]:
    print(' ', os.path.basename(f))

In [ ]:
# Analyze audio durations
durations = []
for f in audio_files[:50]:  # Sample first 50
    try:
        audio, sr = load_audio(f)
        durations.append(len(audio) / sr)
    except:
        pass

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].hist(durations, bins=20, color='#6366f1', edgecolor='white', alpha=0.8)
axes[0].set_title('Audio Duration Distribution', fontsize=13, pad=10)
axes[0].set_xlabel('Duration (seconds)')
axes[0].set_ylabel('Count')

axes[1].bar(['Min', 'Mean', 'Max', 'Total (min)'],
            [min(durations), np.mean(durations), max(durations), sum(durations)/60],
            color=['#f59e0b', '#6366f1', '#10b981', '#ef4444'])
axes[1].set_title('Audio Statistics', fontsize=13, pad=10)

plt.tight_layout()
plt.show()
print(f'Mean duration: {np.mean(durations):.2f}s | Total: {sum(durations)/60:.1f} min')

In [ ]:
# Visualize a sample audio waveform and spectrogram
if audio_files:
    audio, sr = load_audio(audio_files[0])
    fig, axes = plt.subplots(2, 1, figsize=(14, 6))
    
    # Waveform
    t = np.linspace(0, len(audio)/sr, len(audio))
    axes[0].plot(t, audio, color='#6366f1', linewidth=0.5)
    axes[0].set_title(f'Waveform: {os.path.basename(audio_files[0])}', fontsize=12)
    axes[0].set_xlabel('Time (s)'); axes[0].set_ylabel('Amplitude')
    
    # Spectrogram
    D = librosa.amplitude_to_db(np.abs(librosa.stft(audio)), ref=np.max)
    librosa.display.specshow(D, sr=sr, x_axis='time', y_axis='hz', ax=axes[1], cmap='inferno')
    axes[1].set_title('Spectrogram', fontsize=12)
    plt.colorbar(axes[1].collections[0], ax=axes[1], format='%+2.0f dB')
    
    plt.tight_layout()
    plt.show()

## 2. Text Classification Dataset Analysis

In [ ]:
df = load_classification_dataset(config.CLASSIFICATION_DATASET)
if df is not None:
    print(f'Shape: {df.shape}')
    print(f'Columns: {list(df.columns)}')
    display(df.head())

In [ ]:
if df is not None:
    label_col = df.columns[-1]
    label_counts = df[label_col].value_counts()
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    label_counts.plot(kind='bar', ax=axes[0], color='#6366f1', edgecolor='white')
    axes[0].set_title('Label Distribution', fontsize=13)
    axes[0].set_xlabel('Category'); axes[0].set_ylabel('Count')
    axes[0].tick_params(axis='x', rotation=45)
    
    axes[1].pie(label_counts.values, labels=label_counts.index,
                colors=['#6366f1','#f59e0b','#10b981','#ef4444','#3b82f6'],
                autopct='%1.1f%%', startangle=90)
    axes[1].set_title('Label Proportions', fontsize=13)
    
    plt.tight_layout()
    plt.show()

## 3. Padakosha (Vocabulary) Analysis

In [ ]:
pad_df = load_padakosha(config.PADAKOSHA_FILE)
if pad_df is not None:
    print(f'Padakosha entries: {len(pad_df)}')
    print(f'Columns: {list(pad_df.columns)}')
    display(pad_df.head(10))